# PCA on Gene Expression Data — Hands-On Walkthrough

**Dataset:** *Gene Expression Cancer RNA-Seq* — UCI Machine Learning Repository (ID 401).  
Source: Weinstein *et al.* (2013), **The Cancer Genome Atlas Pan-Cancer analysis project**, *Nature Genetics*, 45(10):1113-20.  
Download URL: `https://archive.ics.uci.edu/ml/machine-learning-databases/00401/TCGA-PANCAN-HiSeq-801x20531.tar.gz`

**Dataset at a glance**

| Property | Value |
|---|---|
| Samples (n) | 801 |
| Features (d) | 20 531 genes |
| Classes | 5 cancer types (BRCA, KIRC, COAD, LUAD, PRAD) |
| Values | Log-transformed RSEM expression (RNA-Seq) |
| License | UCI ML Repository — free for research/teaching |

This notebook shows the end-to-end practical application of PCA on real high-dimensional biological data:

1. Load and inspect the dataset.
2. Preprocess: train/test split **before** fitting, variance filter, standardization.
3. Fit PCA with the randomized SVD solver (the right choice at d ≈ 20 000).
4. Use the scree / cumulative-variance curves to pick *k*.
5. Visualize samples in the top-2 and top-3 PC planes.
6. Inspect loadings — which genes drive each PC?
7. Compare downstream classification: **raw features vs PCA features**.
8. Sweep *k* to show the accuracy-vs-compression trade-off.

---
## Cell 1 — Imports & Reproducibility

Fix `RANDOM_STATE` so every split, solver, and classifier is reproducible.

In [ ]:
import os, time, tarfile, urllib.request, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

---
## Cell 2 — Download & Extract the Dataset

The tarball (~50 MB) is cached under `./data/`. Re-running the cell after the first run is a no-op.

The archive contains two CSV files:
- `data.csv`  — 801 rows × 20 532 columns (first column = sample ID, rest = `gene_0` … `gene_20530`).
- `labels.csv` — sample ID and cancer class.

In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00401/TCGA-PANCAN-HiSeq-801x20531.tar.gz'
TARBALL = DATA_DIR / 'TCGA-PANCAN-HiSeq-801x20531.tar.gz'
EXTRACT_DIR = DATA_DIR / 'TCGA-PANCAN-HiSeq-801x20531'

if not EXTRACT_DIR.exists():
    if not TARBALL.exists():
        print('Downloading TCGA Pan-Cancer RNA-Seq dataset (~50 MB)...')
        urllib.request.urlretrieve(URL, TARBALL)
    print('Extracting...')
    with tarfile.open(TARBALL) as tf:
        tf.extractall(DATA_DIR)

print('Files in', EXTRACT_DIR, ':', os.listdir(EXTRACT_DIR))

---
## Cell 3 — Load and Inspect

First checks for any dataset:
- shape
- class balance
- any NaNs?
- summary statistics on a sample of features

In [ ]:
X_df = pd.read_csv(EXTRACT_DIR / 'data.csv', index_col=0)
y_df = pd.read_csv(EXTRACT_DIR / 'labels.csv', index_col=0)

print('Expression matrix :', X_df.shape)
print('Labels            :', y_df.shape)
print('\nClass balance:')
print(y_df['Class'].value_counts())
print('\nAny NaNs in X?', X_df.isna().any().any())
X_df.iloc[:5, :6]

In [ ]:
X_df.sample(5, axis=1, random_state=RANDOM_STATE).describe().T

---
## Cell 4 — Encode Labels & Stratified Train/Test Split

Do the split **before** any fitting (PCA, scaler, variance filter). Otherwise information from the test set leaks into the transformation and test metrics are optimistic.

`stratify=y` preserves class proportions in both splits.

In [ ]:
assert (X_df.index == y_df.index).all(), 'Sample IDs must line up'

le = LabelEncoder()
y = le.fit_transform(y_df['Class'].values)
class_names = le.classes_
print('Classes:', dict(enumerate(class_names)))

X = X_df.values.astype(np.float32)
gene_names = X_df.columns.values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE,
)
print('Train:', X_train.shape, ' Test:', X_test.shape)

---
## Cell 5 — Preprocessing: Variance Filter + Standardization

**Variance filter.** Many genes have near-zero variance across samples and contribute nothing. Dropping them is cheap.

**Standardization.** Gene variances still span orders of magnitude. PCA maximizes variance, so without standardization a few high-variance genes dominate PC1 and hide real structure.

Both steps live inside a `Pipeline`, so the fitted statistics are learned on train and reused on test — no leakage.

In [ ]:
preproc = Pipeline([
    ('var_filter', VarianceThreshold(threshold=0.0)),
    ('scaler',     StandardScaler()),
])

X_train_p = preproc.fit_transform(X_train)
X_test_p  = preproc.transform(X_test)

kept = preproc.named_steps['var_filter'].get_support().sum()
print(f'Genes kept after variance filter : {kept:,} / {X_train.shape[1]:,}')
print(f'Processed train shape             : {X_train_p.shape}')
print(f'Mean ~ 0 ? {X_train_p.mean():+.2e}    Std ~ 1 ? {X_train_p.std():.3f}')

---
## Cell 6 — Fit PCA (Randomized SVD)

With `d ≈ 20 000` and `n ≈ 600` (training), `svd_solver='randomized'` approximates the top-*k* singular vectors in `O(n · d · k)`. The alternative (`svd_solver='full'`) forms a `d × d` covariance matrix — that's 1.6 GB of floats.

We fit with a generous `n_components=100` to study the variance curve, then decide on the *k* we actually need.

In [ ]:
t0 = time.time()
pca_full = PCA(n_components=100, svd_solver='randomized',
               random_state=RANDOM_STATE).fit(X_train_p)
print(f'PCA fit in {time.time() - t0:.2f} s')
print(f'Components computed           : {pca_full.n_components_}')
print(f'Cumulative variance @ 100 PCs : {pca_full.explained_variance_ratio_.sum():.3f}')

---
## Cell 7 — Scree & Cumulative-Variance Plots → Pick *k*

Two diagnostic plots:
1. **Scree plot** (log-y) — individual variance per PC; look for an elbow.
2. **Cumulative explained variance** — pick *k* at your target threshold (e.g. 90 %).

In [ ]:
evr  = pca_full.explained_variance_ratio_
cevr = np.cumsum(evr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(np.arange(1, len(evr) + 1), evr, 'o-', ms=4)
axes[0].set(xlabel='Principal component', ylabel='Explained variance ratio',
            yscale='log', title='Scree plot (log-y)')
axes[0].grid(True, which='both', alpha=0.3)

axes[1].plot(np.arange(1, len(cevr) + 1), cevr, 's-', ms=4, color='tab:orange')
for thr in (0.80, 0.90, 0.95):
    k_thr = int(np.searchsorted(cevr, thr) + 1)
    axes[1].axhline(thr, ls='--', color='grey', alpha=0.5)
    axes[1].axvline(k_thr, ls=':',  color='grey', alpha=0.5)
    axes[1].text(k_thr + 1, thr - 0.02, f'k={k_thr} @ {thr:.0%}', fontsize=9)
axes[1].set(xlabel='Number of components', ylabel='Cumulative explained variance',
            title='Cumulative EVR')
plt.tight_layout(); plt.show()

k_90 = int(np.searchsorted(cevr, 0.90) + 1)
print(f'k for 90 % variance: {k_90}')

We went from **20 531** genes to a few dozen PCs that capture ~90 % of the total variance — roughly **300× compression** with negligible information loss.

---
## Cell 8 — Visualize the Top Principal Components

PCA is unsupervised, so any visible clustering in PC space is emergent — the algorithm never saw the labels. If the five cancer types separate cleanly, that's strong evidence the dominant variance directions align with biology.

In [ ]:
Z_train = pca_full.transform(X_train_p)[:, :3]
Z_test  = pca_full.transform(X_test_p)[:, :3]

fig, ax = plt.subplots(figsize=(7, 6))
palette = sns.color_palette('tab10', n_colors=len(class_names))
for i, cls in enumerate(class_names):
    mask = y_train == i
    ax.scatter(Z_train[mask, 0], Z_train[mask, 1],
               s=28, alpha=0.8, color=palette[i], label=cls, edgecolor='white', lw=0.3)
ax.set(xlabel=f'PC1 ({evr[0]:.1%} var)', ylabel=f'PC2 ({evr[1]:.1%} var)',
       title='TCGA Pan-Cancer samples in the top-2 PCA plane')
ax.legend(loc='best', frameon=True); plt.tight_layout(); plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

fig = plt.figure(figsize=(8, 6))
ax  = fig.add_subplot(111, projection='3d')
for i, cls in enumerate(class_names):
    mask = y_train == i
    ax.scatter(Z_train[mask, 0], Z_train[mask, 1], Z_train[mask, 2],
               s=22, alpha=0.8, color=palette[i], label=cls)
ax.set(xlabel=f'PC1 ({evr[0]:.1%})',
       ylabel=f'PC2 ({evr[1]:.1%})',
       zlabel=f'PC3 ({evr[2]:.1%})',
       title='Top-3 PCA space')
ax.legend(); plt.tight_layout(); plt.show()

---
## Cell 9 — Interpret Loadings: Top Genes per PC

Each column of `pca.components_` is a loading vector of length *d*. Large |loading| means that gene contributes strongly to that PC.

*Caveat:* large loading ≠ "most important gene for classification". It means *largest contribution to the direction of maximum variance*. Still useful as a sanity check.

In [ ]:
kept_mask  = preproc.named_steps['var_filter'].get_support()
kept_genes = gene_names[kept_mask]
loadings   = pca_full.components_

TOP = 15
for pc in range(3):
    idx = np.argsort(np.abs(loadings[pc]))[::-1][:TOP]
    top = pd.DataFrame({
        'gene':    kept_genes[idx],
        'loading': loadings[pc, idx].round(4),
    })
    print(f'\nTop {TOP} genes for PC{pc + 1} (explains {evr[pc]:.2%} variance):')
    print(top.to_string(index=False))

---
## Cell 10 — Downstream Classification: Raw vs PCA Features

Compare two pipelines on identical splits:
1. **Raw** — StandardScaler → Logistic Regression.
2. **PCA** — StandardScaler → PCA(90 % variance) → Logistic Regression.

Measure test accuracy and wall-clock fit time.

In [ ]:
def bench(pipe, name):
    t0 = time.time(); pipe.fit(X_train, y_train)
    fit_t = time.time() - t0
    t0 = time.time(); preds = pipe.predict(X_test)
    pred_t = time.time() - t0
    acc = accuracy_score(y_test, preds)
    print(f'{name:18s} | acc = {acc:.4f} | fit = {fit_t:5.2f}s | predict = {pred_t:5.3f}s')
    return preds, acc, fit_t

pipe_raw = Pipeline([
    ('var_filter', VarianceThreshold(0.0)),
    ('scaler',     StandardScaler()),
    ('clf',        LogisticRegression(max_iter=2000, n_jobs=-1,
                                      random_state=RANDOM_STATE)),
])

pipe_pca = Pipeline([
    ('var_filter', VarianceThreshold(0.0)),
    ('scaler',     StandardScaler()),
    ('pca',        PCA(n_components=0.90, svd_solver='randomized',
                       random_state=RANDOM_STATE)),
    ('clf',        LogisticRegression(max_iter=2000, n_jobs=-1,
                                      random_state=RANDOM_STATE)),
])

preds_raw, acc_raw, t_raw = bench(pipe_raw, 'Raw features')
preds_pca, acc_pca, t_pca = bench(pipe_pca, 'PCA features')

print(f'\nSpeed-up from PCA pipeline: {t_raw / max(t_pca, 1e-9):.1f}×')
print(f'Components actually retained : {pipe_pca.named_steps["pca"].n_components_}')

In [ ]:
print('Classification report (PCA pipeline)')
print(classification_report(y_test, preds_pca, target_names=class_names))

cm = confusion_matrix(y_test, preds_pca)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set(xlabel='Predicted', ylabel='True', title='Confusion matrix — PCA + LogReg')
plt.tight_layout(); plt.show()

---
## Cell 11 — Accuracy vs Number of Components

Sweep *k* over a range and plot (a) cumulative variance and (b) downstream classifier accuracy. Visual justification for the *k* you pick.

In [ ]:
ks, accs, var_kept = [], [], []
for k in [2, 5, 10, 20, 40, 60, 100]:
    p = Pipeline([
        ('var_filter', VarianceThreshold(0.0)),
        ('scaler',     StandardScaler()),
        ('pca',        PCA(n_components=k, svd_solver='randomized',
                           random_state=RANDOM_STATE)),
        ('clf',        LogisticRegression(max_iter=2000, n_jobs=-1,
                                          random_state=RANDOM_STATE)),
    ]).fit(X_train, y_train)
    ks.append(k)
    accs.append(accuracy_score(y_test, p.predict(X_test)))
    var_kept.append(p.named_steps['pca'].explained_variance_ratio_.sum())

fig, ax1 = plt.subplots(figsize=(7, 4))
l1, = ax1.plot(ks, accs, 'o-', color='tab:blue',  label='Test accuracy')
ax1.set(xlabel='k (principal components)', ylabel='Test accuracy')
ax2 = ax1.twinx()
l2, = ax2.plot(ks, var_kept, 's--', color='tab:orange', label='Cumulative var')
ax2.set_ylabel('Cumulative explained variance')
ax1.legend(handles=[l1, l2], loc='lower right')
plt.title('Accuracy and variance vs k'); plt.tight_layout(); plt.show()

---
## Cell 12 — Summary

- Loaded a real, publicly hosted high-dimensional biology dataset (TCGA Pan-Cancer RNA-Seq, 801 × 20 531).
- Split **before** any fitting, filtered zero-variance genes, standardized — no leakage.
- Fit randomized-SVD PCA; retained ~90 % variance with a few dozen components (~300× compression).
- Five cancer types separated cleanly in the top-2 PC plane, purely from unsupervised variance maximization.
- Inspected loadings to identify genes dominating each PC.
- Benchmarked logistic regression with and without PCA — same or better accuracy, much faster training.

This is the canonical PCA workflow any time `d ≫ n`: variance filter → standardize → randomized SVD → pick *k* by explained variance or downstream CV → feed PCs into the next model.